# Generate the Data

In [1]:
"""
01_generate_data.py
Generates a synthetic but realistic loan applicant dataset
with intentional real-world messiness for a Data Analyst portfolio project.
"""

import numpy as np
import pandas as pd

np.random.seed(42)
N = 6000

# ---- Core applicant fields ----
applicant_id = [f"APP{str(i).zfill(6)}" for i in range(1, N + 1)]

genders = np.random.choice(["Male", "Female"], size=N, p=[0.58, 0.42])

# Income: right-skewed (lognormal) -- realistic for income data
income = np.random.lognormal(mean=10.8, sigma=0.55, size=N).round(0)

# Employment type with inconsistent casing (real-world mess)
emp_raw = np.random.choice(
    ["Salaried", "salaried", "Self-Employed", "self employed", "SELF-EMPLOYED", "Salaried "],
    size=N, p=[0.35, 0.1, 0.2, 0.1, 0.1, 0.15]
)

regions = np.random.choice(
    ["North", "South", "East", "West", "north", "EAST", "West "],
    size=N, p=[0.22, 0.22, 0.18, 0.18, 0.07, 0.07, 0.06]
)

credit_history = np.random.choice([1, 0, np.nan], size=N, p=[0.78, 0.12, 0.10])  # 1=good,0=bad

loan_amount = np.random.lognormal(mean=11.5, sigma=0.6, size=N).round(-3)
loan_term_months = np.random.choice([12, 24, 36, 48, 60, 84, 120, 180, 240, 360],
                                      size=N, p=[.05,.07,.10,.10,.15,.13,.15,.10,.08,.07])

existing_emi = (income * np.random.uniform(0, 0.35, size=N)).round(0)

age = np.random.normal(38, 11, size=N).clip(21, 70).round(0)

dependents = np.random.choice([0, 1, 2, 3, 4], size=N, p=[0.35, 0.25, 0.2, 0.13, 0.07])

married = np.random.choice(["Yes", "No"], size=N, p=[0.62, 0.38])

education = np.random.choice(["Graduate", "Not Graduate"], size=N, p=[0.72, 0.28])

application_date = pd.to_datetime("2023-01-01") + pd.to_timedelta(
    np.random.randint(0, 730, size=N), unit="D"
)

# ---- Default probability logic (ground truth signal, not pure noise) ----
# Lower credit history, higher EMI burden, lower income -> higher default risk
emi_to_income = np.where(income > 0, existing_emi / (income + 1), 0)
base_risk = (
    0.35 * (1 - np.nan_to_num(credit_history, nan=0.5))
    + 0.30 * np.clip(emi_to_income, 0, 1)
    + 0.15 * (loan_amount / loan_amount.max())
    + 0.10 * (dependents / 4)
    - 0.10 * (education == "Graduate").astype(int)
)
prob_default = np.clip(base_risk + np.random.normal(0, 0.08, size=N), 0.02, 0.95)
default_status = np.random.binomial(1, prob_default)

df = pd.DataFrame({
    "applicant_id": applicant_id,
    "gender": genders,
    "married": married,
    "dependents": dependents,
    "education": education,
    "employment_type": emp_raw,
    "applicant_income": income,
    "existing_emi": existing_emi,
    "loan_amount": loan_amount,
    "loan_term_months": loan_term_months,
    "credit_history": credit_history,
    "region": regions,
    "age": age,
    "application_date": application_date,
    "default_status": default_status,
})

# ---- Inject real-world mess ----

# 1. Missing values in a few more columns
for col, frac in [("applicant_income", 0.03), ("loan_amount", 0.02), ("gender", 0.015), ("age", 0.01)]:
    idx = np.random.choice(df.index, size=int(N * frac), replace=False)
    df.loc[idx, col] = np.nan

# 2. Duplicate rows (some exact, some near-duplicate with same applicant_id)
dupe_idx = np.random.choice(df.index, size=80, replace=False)
df = pd.concat([df, df.loc[dupe_idx]], ignore_index=True)

# 3. Outliers
out_idx = np.random.choice(df.index, size=15, replace=False)
df.loc[out_idx, "applicant_income"] = df.loc[out_idx, "applicant_income"] * np.random.uniform(15, 30)

out_idx2 = np.random.choice(df.index, size=10, replace=False)
df.loc[out_idx2, "age"] = np.random.choice([1, 2, 150, 200], size=10)

# 4. Shuffle rows so it doesn't look generated in order
df = df.sample(frac=1, random_state=7).reset_index(drop=True)

df.to_csv("loan_applicants_raw.csv", index=False)

print("Shape:", df.shape)
print("\nMissing values per column:\n", df.isna().sum())
print("\nDuplicate applicant_ids:", df['applicant_id'].duplicated().sum())
print("\nSample:\n", df.head(8))
print("\nDefault rate:", df['default_status'].mean().round(3))

Shape: (6080, 15)

Missing values per column:
 applicant_id          0
gender               91
married               0
dependents            0
education             0
employment_type       0
applicant_income    184
existing_emi          0
loan_amount         122
loan_term_months      0
credit_history      573
region                0
age                  62
application_date      0
default_status        0
dtype: int64

Duplicate applicant_ids: 80

Sample:
   applicant_id  gender married  dependents     education employment_type  \
0    APP004913    Male     Yes           2      Graduate   Self-Employed   
1    APP002130    Male      No           0  Not Graduate   self employed   
2    APP003816    Male     Yes           0      Graduate        Salaried   
3    APP002894  Female      No           0      Graduate        Salaried   
4    APP000446  Female      No           3      Graduate        Salaried   
5    APP001990  Female     Yes           2      Graduate        salaried   
6    APP0

# Load the Data

In [2]:
df = pd.read_csv("loan_applicants_raw.csv")
df

,applicant_id,gender,married,dependents,education,employment_type,applicant_income,existing_emi,loan_amount,loan_term_months,credit_history,region,age,application_date,default_status
0,APP004913,Male,Yes,2,Graduate,Self-Employed,43510.0,767.0,398000.0,60,1.0,South,39.0,2023-06-20,0
1,APP002130,Male,No,0,Not Graduate,self employed,135336.0,10906.0,94000.0,180,1.0,EAST,44.0,2024-01-05,0
2,APP003816,Male,Yes,0,Graduate,Salaried,93748.0,423.0,75000.0,36,1.0,EAST,33.0,2024-04-14,0
3,APP002894,Female,No,0,Graduate,Salaried,77999.0,9687.0,207000.0,48,1.0,South,40.0,2023-07-18,0
4,APP000446,Female,No,3,Graduate,Salaried,17692.0,4966.0,42000.0,360,1.0,South,36.0,2023-11-19,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6075,APP005700,Male,Yes,0,Graduate,Self-Employed,86271.0,12790.0,140000.0,360,1.0,North,53.0,2023-06-07,0
6076,APP002551,Male,No,0,Graduate,salaried,144611.0,17724.0,27000.0,84,1.0,West,44.0,2024-09-21,0
6077,APP000538,Male,Yes,0,Graduate,Salaried,73004.0,20337.0,195000.0,36,1.0,North,41.0,2023-03-07,0
6078,APP001221,Female,Yes,1,Graduate,Salaried,33054.0,2064.0,108000.0,180,1.0,East,52.0,2024-05-23,0


# Start Cleaning the Data (Data Cleaning)

## ---- Step 1: Remove duplicates ----

In [3]:
print("Before Cleaning", df.shape)

Before Cleaning (6080, 15)


In [4]:
# These are duplicate applicant_ids (same applicant appearing twice due to
# simulated re-submission glitches). Keep the first occurrence.

df = df.drop_duplicates(subset = "applicant_id", keep = "first")
print("After Removing Duplicates", df.shape)

After Removing Duplicates (6000, 15)


## ---- Step 2: Standardize text columns ----

In [5]:
# employment_type: strip whitespace, lowercase, then map to clean categories
df["employment_type"] = df["employment_type"].str.strip().str.lower()
df["employment_type"] = df["employment_type"].replace({
    "salaried" : "Salaried",
    "self employed" : "Self-Employed",
    "self-employed" : "Self-Employed",
})

# region: same pattern - strip, lowercase, then capitalize properly
df["region"] = df["region"].str.strip().str.lower()
df["region"] = df["region"].replace ({
    "north" : "North",
    "south" : "South",
    "east" : "East",
    "west" : "West",
})

# Verify the cleanup worked
print("\nemployment_type unique values: ", df["employment_type"].unique())
print("region unique values: ", df["region"].unique())


employment_type unique values:  ['Self-Employed' 'Salaried']
region unique values:  ['South' 'East' 'West' 'North']


C:\Users\soumi\AppData\Local\Temp\ipykernel_20536\43545362.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["employment_type"] = df["employment_type"].str.strip().str.lower()
C:\Users\soumi\AppData\Local\Temp\ipykernel_20536\43545362.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["employment_type"] = df["employment_type"].replace({
C:\Users\soumi\AppData\Local\Temp\ipykernel_20536\43545362.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try us

In [6]:
df["employment_type"].value_counts()

employment_type
Salaried         3548
Self-Employed    2452
Name: count, dtype: int64

In [7]:
df["region"].value_counts()

region
North    1662
West     1509
East     1508
South    1321
Name: count, dtype: int64

In [8]:
# drop_duplicates(subset="applicant_id") rather than just drop_duplicates() — this is deliberate.
# If two rows had the same applicant_id but slightly different data due to a re-entry, a plain drop_duplicates()
# wouldn't catch it since it only removes fully identical rows. Always dedupe on the business key (applicant_id), not the whole row.


# The .str.strip().str.lower() then .replace() pattern is the standard fix for messy categorical text — normalize first,
# then map to your final clean labels. Doing .replace() directly on the raw inconsistent values would mean writing out every casing variant by hand.


# Try running this, then check df["employment_type"].value_counts() and df["region"].value_counts() to confirm there's exactly
# one clean category per group.

## ---- Step 3: Fixing invalid ages ----

In [9]:
# find invalid ages and convert to NaN
invalid_mask = (df["age"] < 18) | (df["age"] > 75)
print("Invalid ages found: ", invalid_mask.sum())

# replace the inavlid ages with NaN
df.loc[invalid_mask, "age"] = np.nan 
print("Missing ages after fix : ", df["age"].isna().sum())
print(df["age"].describe())

Invalid ages found:  10
Missing ages after fix :  70
count    5930.000000
mean       38.485160
std        10.315557
min        21.000000
25%        31.000000
50%        38.000000
75%        46.000000
max        70.000000
Name: age, dtype: float64


## ---- Step 4: Handle missing values ----

In [10]:
# applicant_income & loan_amount: median imputation (robust to skew/outliers)
df["applicant_income"] = df["applicant_income"].fillna(df["applicant_income"].median())
df["loan_amount"] = df["loan_amount"].fillna(df["loan_amount"].median())

# age: median is fine here too (numeric, no strong skew after outlier fix)
df["age"] = df["age"].fillna(df["age"].median())

# gender: mode imputation
df["gender"] = df["gender"].fillna(df["gender"].mode()[0])

# credit_history: convert to a 3-category meaningful field instead of guessing
df["credit_history"] = df["credit_history"].map({1.0: "Good", 0.0: "Bad"})
df["credit_history"] = df["credit_history"].fillna("No History")

# confirm no missing values remain anywhere
print(df.isna().sum())

applicant_id        0
gender              0
married             0
dependents          0
education           0
employment_type     0
applicant_income    0
existing_emi        0
loan_amount         0
loan_term_months    0
credit_history      0
region              0
age                 0
application_date    0
default_status      0
dtype: int64


C:\Users\soumi\AppData\Local\Temp\ipykernel_20536\3392862233.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["applicant_income"] = df["applicant_income"].fillna(df["applicant_income"].median())
C:\Users\soumi\AppData\Local\Temp\ipykernel_20536\3392862233.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["loan_amount"] = df["loan_amount"].fillna(df["loan_amount"].median())
C:\Users\soumi\AppData\Local\Temp\ipykernel_20536\3392862233.py:6: SettingWithCopyWarning: 
A value is trying to be set on a 

## ---- Step 5: Handle income outliers ----

In [11]:
Q1 = df["applicant_income"].quantile(0.25)
Q3 = df["applicant_income"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1 : {Q1}, Q3 : {Q3}, IQR : {IQR}")
print(f"Lower Bound : {lower_bound}")
print(f"Upper Bound : {upper_bound}")

# See how many outliers exist
outliers = df[(df["applicant_income"] < lower_bound) | (df["applicant_income"] > upper_bound)]
print(f"\nNumber of Outliers detected: {len(outliers)}")
print(outliers["applicant_income"].sort_values(ascending = False).head(10))

Q1 : 34098.0, Q3 : 69987.0, IQR : 35889.0
Lower Bound : -19735.5
Upper Bound : 123820.5

Number of Outliers detected: 268
2221    3.558928e+06
6069    2.784470e+06
3408    2.498536e+06
1794    1.666170e+06
2191    1.438169e+06
5606    1.286395e+06
1095    1.231046e+06
1234    9.687431e+05
185     9.649427e+05
1509    8.932462e+05
Name: applicant_income, dtype: float64


In [12]:
# ---- Why Log-Transformed IQR instead of Standard IQR? ----
# The standard IQR method above detected 268 outliers, but we only injected ~15 bad values.
# This over-detection happens because IQR assumes roughly symmetric data.
# applicant_income is RIGHT-SKEWED (lognormal) -- most people earn moderately,
# a few earn very high. On skewed data, the upper fence (Q3 + 1.5*IQR) sits
# too close to the bulk of the data, incorrectly flagging legitimate high earners.
#
# Fix: Apply IQR on log(income) instead of raw income.
# np.log() compresses the long right tail, making the distribution roughly symmetric.
# Now IQR fences are placed more fairly, catching only the truly absurd values.
# After detection, we convert bounds back to original scale using np.exp().
#
# Winsorizing (Option A / clipping) is chosen over dropping because:
# - Only the income value is bad, the rest of that row (age, loan_amount, etc.) is valid
# - Dropping would unnecessarily lose good data
# - Capping to the upper bound is a fair, conservative correction

In [13]:
import numpy as np

# Apply IQR on log-transformed income (handles right skew properly)
log_income = np.log(df["applicant_income"])

Q1 = log_income.quantile(0.25)
Q3 = log_income.quantile(0.75)
IQR = Q3 - Q1 

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Convert bounds back to original scale
lower_bound_original = np.exp(lower_bound)
upper_bound_original = np.exp(upper_bound)

print(f"Lower Bound (original scale): {lower_bound_original:.0f}")
print(f"Upper Bound (original scale): {upper_bound_original:.0f}")

# Check how many get flagged now
outlier_mask = (df["applicant_income"] < lower_bound_original) | (df["applicant_income"] > upper_bound_original)
print(f"Outliers detected: {outlier_mask.sum()}")

Lower Bound (original scale): 11596
Upper Bound (original scale): 205802
Outliers detected: 73


In [14]:
# Option A — Cap (Winsorize): Replace outlier values with the upper/lower bound.
# You keep the row but clip the extreme value. Best when the row's other data is valid and you don't want to lose it.
# Code:
# df["applicant_income"] = df["applicant_income"].clip(lower=lower_bound, upper=upper_bound)

# Option B — Drop: Remove the outlier rows entirely. Best when the entire row is suspicious, not just one column.
# Code:
# df = df[(df["applicant_income"] >= lower_bound) & (df["applicant_income"] <= upper_bound)]

# Option C — Flag and keep: Add a new column marking the row as an outlier but don't touch the value.
# Best when you want to analyze outliers separately later.
# Code:
# df["income_outlier_flag"] = ((df["applicant_income"] < lower_bound) | (df["applicant_income"] > upper_bound)).astype(int)

In [15]:
# Cap the outliers (Winsorize) -- keeps the row, just clips the extreme value
# We're choosing Option A because the rest of the row's data is valid

df["applicant_income"] = df["applicant_income"].clip(
    lower = lower_bound_original,
    upper = upper_bound_original
)
print("Income after capping: ")
print(df["applicant_income"].describe())

Income after capping: 
count      6000.000000
mean      56719.895997
std       33409.648553
min       11595.680385
25%       34098.000000
50%       48644.000000
75%       69987.000000
max      205802.187190
Name: applicant_income, dtype: float64


C:\Users\soumi\AppData\Local\Temp\ipykernel_20536\632218529.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["applicant_income"] = df["applicant_income"].clip(


## ---- Step 6: Fix data types ----

In [16]:
print(df.dtypes)

applicant_id         object
gender               object
married              object
dependents            int64
education            object
employment_type      object
applicant_income    float64
existing_emi        float64
loan_amount         float64
loan_term_months      int64
credit_history       object
region               object
age                 float64
application_date     object
default_status        int64
dtype: object


In [17]:
# 1. application_date should be datetime, not a plain string
df["application_date"] = pd.to_datetime(df["application_date"])

# 2. age should be int (no one is 34.0 years old)
# We made it float earlier when we introduced NaN, then filled it
# Now that there are no nulls, we can safely cast to int
df["age"] = df["age"].astype(int)

# 3. Convert categorical text columns to pandas category dtype
# Saves memory and makes groupby operations faster
for col in ["gender", "married", "education", "employment_type", "region", "credit_history"]:
    df[col] = df[col].astype("category")

# Verify
print(df.dtypes)

applicant_id                object
gender                    category
married                   category
dependents                   int64
education                 category
employment_type           category
applicant_income           float64
existing_emi               float64
loan_amount                float64
loan_term_months             int64
credit_history            category
region                    category
age                          int32
application_date    datetime64[ns]
default_status               int64
dtype: object


C:\Users\soumi\AppData\Local\Temp\ipykernel_20536\2963868263.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["application_date"] = pd.to_datetime(df["application_date"])
C:\Users\soumi\AppData\Local\Temp\ipykernel_20536\2963868263.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["age"] = df["age"].astype(int)
C:\Users\soumi\AppData\Local\Temp\ipykernel_20536\2963868263.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,c

## Final Cleaning Checkpoint

In [18]:
print("Shape: ", df.shape)
print("\nMissing Values:\n", df.isna().sum())
print("\nEmployment Type:\n", df["employment_type"].value_counts())
print("\nRegion:\n", df["region"].value_counts())
print("\nCredit History:\n", df["credit_history"].value_counts())
print("\nAge Range:\n", df["age"].min(), "-", df["age"].max())
print("\nIncome Range:\n", df["applicant_income"].min(), "-", df["applicant_income"].max())

Shape:  (6000, 15)

Missing Values:
 applicant_id        0
gender              0
married             0
dependents          0
education           0
employment_type     0
applicant_income    0
existing_emi        0
loan_amount         0
loan_term_months    0
credit_history      0
region              0
age                 0
application_date    0
default_status      0
dtype: int64

Employment Type:
 employment_type
Salaried         3548
Self-Employed    2452
Name: count, dtype: int64

Region:
 region
North    1662
West     1509
East     1508
South    1321
Name: count, dtype: int64

Credit History:
 credit_history
Good          4720
Bad            711
No History     569
Name: count, dtype: int64

Age Range:
 21 - 70

Income Range:
 11595.680385495243 - 205802.187189847


# Save Cleaned Dataset

In [19]:
df.to_csv("loan_applicants_clean.csv", index = False)
print("Clean dataset saved. Shape: ", df.shape)

Clean dataset saved. Shape:  (6000, 15)
